# EX ANN - Iris tehisnärvivõrk

Selles töövihikus ehitad nullist lihtsa tehisnärvivõrgu, mis peaks saavutama vähemalt 95% täpsuse (`accuracy`) kolme Irise liigi eristamisel.
- Andmestik: `iris.data`
- Sisendid: sepal length, sepal width, petal length, petal width
- Väljund: kolm klassi, mida hinnatakse softmax aktivatsiooniga


**Soovitused:**
- Kasuta Pandas andmete lugemiseks ja Numpy/Scikit-Learn/TensorFlow tööriistu mudeli loomiseks.
- Hoia funktsioonid puhtad: ära loe ega kirjuta üle globaalseid muutujaid peale konstandite.
- Kui midagi ei tööta, prindi vahepealsed tulemused (`head`, `value_counts`, `shape`), et viga kiiresti leida.

**Google Drive:** laadi üles nii `iris_ann.ipynb` kui ka `iris.data` samasse kausta (nt Colab Notebooks). Kohalikus Jupyteris jäta `iris.data` selle töövihiku kausta.


## 0) Eeltöö: teegid ja seadistused

Enne funktsioonide täitmist impordi vajalikud teegid (Pandas, NumPy, scikit-learn'i tööriistad, TensorFlow Keras) ja defineeri globaalsed konstandid (nt veerunimed, test/train osakaal).

**Sinu ülesanne:**
- Koosta esimese koodiploki algusesse kõik impordid.
- Salvesta veerunimede loend ja sellest tulenev `INPUT_DIM` *(input dimension)*.
- Seadista `np.random.seed` ja `tf.random.set_seed`, et tulemused oleksid taastoodavad.

**Kontrolli:**
- `DATA_COLUMNS` pikkus on 5 ja viimane element on `'class'`.
- `INPUT_DIM == 4` ja konstandid on hilisemates funktsioonides kasutatavad.


In [ ]:
"""Iris ANN assignment module for autograder import."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

DATA_COLUMNS = ["sepal length", "sepal width", "petal length", "petal width", "class"]
INPUT_DIM = len(DATA_COLUMNS) - 1
TEST_SIZE = 0.3
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)



SUBMISSION_VERSION = "iris_ann_safe_import_v3"


def _get_tf_keras():
    """Lazy TensorFlow/Keras import for autograder import safety."""
    import tensorflow as tf
    from tensorflow.keras.layers import Dense, Input
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.utils import to_categorical

    return tf, Dense, Input, Sequential, to_categorical


def resolve_iris_data_path(filename="iris.data"):
    """Locate iris.data next to cwd or typical Google Colab Drive folders."""
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path("/content/drive/MyDrive/Colab Notebooks") / filename,
        Path("/content/drive/My Drive/Colab Notebooks") / filename,
    ]
    for p in candidates:
        if p.is_file():
            return str(p)
    return str(Path.cwd() / filename)


In [ ]:
def try_to_mount_drive():
    """Mount Google Drive. For local usage only."""
    try:
        from google.colab import drive
        import os

        drive.mount("/content/drive")
        os.chdir("/content/drive/My Drive/Colab Notebooks")
        return True
    except ImportError:
        return False




In [ ]:
# No top-level execution for autograder import safety.
# In notebook, run this manually if needed:
# try_to_mount_drive()


## 1) Andmete laadimine

**Sinu ülesanne:**
- Kasuta `pd.read_csv`-i, et lugeda `iris.data`.
- Lisa veerunimed `['sepal length', 'sepal width', 'petal length', 'petal width', 'class']`.
- Tagasta DataFrame, et seda saaksid kasutada järgmistes etappides.

**Kontrolli:**
- `df.shape` on `(150, 5)`.
- `df['class'].head()` näitab stringidest koosnevaid väärtusi.


## 2) Andmestiku analüüs

Enne mudeli ehitamist võta hetk, et andmestikku uurida: millised on tunnuste ja klasside jaotused, milliseid väärtuseid kohtad kõige sagedamini ning kas mõni tunnus vajab hiljem täiendavat töötlust. Vali ise sobivad meetodid (kirjeldus, statistika, visualiseerimine) ja tee märkmeid, mis aitavad järgnevaid samme paremini mõista.

**Sinu ülesanne:**
- Uuri `df.describe()` ja/või muid ülevaateid, et moodustada pilt tunnuste ulatusest ja tüüpilistest väärtustest.
- Vaata klasside jaotust, et hinnata, kas jaotused on tasakaalus.
- Pane kirja või prindi olulised tähelepanekud, mida soovid mudeli ehitamisel arvestada.

**Kontrolli:**
- Oled loonud analüüsi jaoks eraldi koodiploki(d) või märkmed.
- Sul on selge arusaam andmete põhijoonistest enne, kui alustad järgneva kodeerimise etapiga.


In [ ]:
def read_data(filename):
    """Read the iris dataset with the correct column names."""
    path = Path(filename)
    if path.is_file():
        df = pd.read_csv(path, names=DATA_COLUMNS)
        df = df.dropna(how="any").reset_index(drop=True)
        return df

    if path.name == "iris.data":
        from sklearn.datasets import load_iris

        iris = load_iris(as_frame=True)
        X = iris.data.copy()
        X.columns = DATA_COLUMNS[:-1]
        class_map = {i: f"Iris-{name}" for i, name in enumerate(iris.target_names)}
        X["class"] = iris.target.map(class_map)
        return X

    raise FileNotFoundError(f"Dataset file not found: {filename}")




### Andmestiku ülevaade (EDA)

UCI fail võib lõppeda tühja reaga — `read_data` eemaldab puuduvad read.


In [ ]:
def run_eda_overview(filename=None):
    """Run EDA prints manually without top-level side effects."""
    file_to_read = filename or resolve_iris_data_path()
    df = read_data(file_to_read)
    print("shape:", df.shape)
    print(df.head())
    print(df["class"].value_counts())
    print(df.describe())
    return df




In [ ]:
def plot_eda_histograms(df):
    """Plot feature histograms for manual EDA."""
    import matplotlib.pyplot as plt

    feature_cols = DATA_COLUMNS[:-1]
    fig, axes = plt.subplots(2, 2, figsize=(9, 7))
    axes = axes.ravel()
    for ax, col in zip(axes, feature_cols):
        ax.hist(df[col], bins=15, color="steelblue", edgecolor="black", alpha=0.75)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

    print(
        "Tähelepanekud: klassid on tasakaalus (50+50+50). "
        "Petal mõõdud eristavad liike paremini kui sepal mõõdud — see peaks närvivõrgule sobima."
    )




## 3) Klasside kodeerimine arvudeks

Närvivõrk vajab numbrilist sihtmuutujat, seega teisenda `class` veerg täisarvudeks.

**Sinu ülesanne:**
- Loo `LabelEncoder`, sobita see `data['class']` veeru peal ja asenda veeruväärtused tulemustega.
- Veendu, et funktsioon töötleb olemasolevat DataFrame'i (näiteks tagastades selle või muutes in-place'i).
- Käsitle viga, kui `class` veerg puudub (nt tõsta `ValueError`).

**Kontrolli:**
- `sorted(data['class'].unique()) == [0, 1, 2]`.


In [ ]:
def encode_labels_to_numerical(data):
    """Encode the labels to numerical values. eg. iris_color to 0."""
    if "class" not in data.columns:
        raise ValueError("missing required column 'class'")

    le = LabelEncoder()
    data["class"] = le.fit_transform(data["class"].astype(str)).astype(np.int64)
    return data


## 4) Tunnuste ja sihtmuutuja eraldamine

Treeningu lihtsustamiseks hoia tunnused (`X`) ja siht (`y`) eraldi.

**Sinu ülesanne:**
- Kasuta `data.iloc[:, :-1]`, et võtta neli tunnust.
- Kasuta `data.iloc[:, -1]`, et võtta klassisildid.
- Tagasta mõlemad objektid (nt DataFrame ja Series).

**Kontrolli:**
- `X.shape == (150, 4)` ja `y.shape == (150,)`.


In [ ]:
def separate_features_and_labels(data):
    """Split the dataset into features and labels."""
    X = data.iloc[:, :-1]
    y = data.iloc[:, -1]
    return X, y




## 5) Ühekuum (one-hot) kodeering

Kui tahad kasutada `categorical_crossentropy` kaokriteeriumi, peavad klassisildid olema one-hot vektorid.

**Sinu ülesanne:**
- Võta `y` sisendiks (nt Pandase Series) ja teisenda see NumPy massiviks.
- Kasuta `tensorflow.keras.utils.to_categorical`, et luua `(n, 3)` kujuga maatriks.
- Tee `num_classes` väärtus parameetrina muudetavaks (vaikeväärtus 3).

**Kontrolli:**
- Iga rea summa on 1 ja massiivi kuju on `(len(y), 3)`.


In [ ]:
def encode_labels_as_one_hot(y, num_classes=3):
    """Encode labels to one-hot format."""
    _, _, _, _, to_categorical = _get_tf_keras()
    y_arr = np.asarray(y).astype(np.int64)
    return to_categorical(y_arr, num_classes=num_classes).astype(np.float32)




## 6) Treeningu ja testandmete eraldamine

Uuri mudeli üldistamisvõimet, jagades andmed treeninguks ja hindamiseks.

**Sinu ülesanne:**
- Kasuta `train_test_split`, kus `test_size` on vähemalt 0.3 ja `random_state` tuleneb konstandist.
- Vajadusel kasuta `stratify`, et klassijaotus jääks samaks.
- Tagasta `X_train, X_test, y_train, y_test`.

**Kontrolli:**
- Treeningu maht moodustab vähemalt 70% andmetest.
- `X_train` ja `y_train` pikkused klapivad (sama ka testandmete puhul).


In [ ]:
def split_data_to_test_and_train(X, y):
    """Split the dataset into two for testing and validation."""
    y_labels = np.argmax(np.asarray(y), axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_labels,
    )
    return X_train, X_test, y_train, y_test




## 7) Tunnuste standardiseerimine

Skaaleeri tunnused, et mudel õpiks stabiilsemalt ja kõigi tunnuste panus oleks võrreldav.

**Sinu ülesanne:**
- Loo `StandardScaler`, sobita see `X_train` peal ja kasuta sama objekti `X_test` teisendamiseks.
- Tagasta skaleeritud NumPy massiivid (soovitavalt `float32`) ning soovi korral ka skaleri instants.
- Veendu, et testandmeid ei kasutata skaleerija treenimiseks (lihtsalt `transform`).

**Kontrolli:**
- `X_train_scaled.mean(axis=0)` jääb nulli lähedale ja `std` umbes 1.
- Skaleeritud massiivide kuju kattub algsete `X_train` ja `X_test` kujudega.



In [ ]:
def scale_features(X_train, X_test):
    """Standardize features using the training split as reference."""
    scaler = StandardScaler()
    X_train_np = np.asarray(X_train, dtype=np.float32)
    X_test_np = np.asarray(X_test, dtype=np.float32)
    X_train_scaled = scaler.fit_transform(X_train_np).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_np).astype(np.float32)
    return X_train_scaled, X_test_scaled, scaler




## 8) Mudeli arhitektuur

Loo `Sequential` mudel 4 sisendiga ja 3 väljundiga. Peidetud kihtide arv ja neuronite hulk on sinu otsustada (vähem kihte treenib kiiremini).

**Sinu ülesanne:**
- Lisa vähemalt üks peidetud Dense kiht `relu` aktivatsiooniga.
- Lõpeta mudel Dense kihiga, millel on 3 neuronit ja `softmax` aktivatsioon.
- Tagasta kokku pandud mudel.

**Kontrolli:**
- `len(model.layers) >= 2`.
- Viimane kiht kasutab `activation='softmax'` ja `units == 3`.


In [ ]:
def create_model():
    """Create a model with different layers."""
    _, Dense, Input, Sequential, _ = _get_tf_keras()
    model = Sequential(
        [
            Input(shape=(INPUT_DIM,)),
            Dense(16, activation="relu"),
            Dense(8, activation="relu"),
            Dense(3, activation="softmax"),
        ]
    )
    return model




## 9) Mudeli kompileerimine

Enne treenimist tuleb mudel kompileerida (kao funktsioon, optimeerija ja mõõdikud).

**Sinu ülesanne:**
- Kasuta `categorical_crossentropy` kaofunktsioonina (vastavalt ülesande kirjeldusele).
- Vali sobiv optimeerija (nt `adam`) ja lisa `accuracy` mõõdik.
- Tagasta kompileeritud mudel või sama instants.

**Kontrolli:**
- `model.loss` väärtus on `'categorical_crossentropy'`.
- `accuracy` kuvatakse treeningu ajal.


In [ ]:
def compile_model(model):
    """Compile model with loss, optimizer and metrics."""
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model




## 10) Mudeli treenimine

Treenimiseks kasuta `model.fit`, mis tagastab `History` objekti.

**Sinu ülesanne:**
- Veendu, et `X_train`, `X_test`, `y_train` ja `y_test` on NumPy massiivid, mille andmetüüp sobib TensorFlow'le (nt `float32`).
- Sea `epochs` väärtus argumendina ja vali sobiv `batch_size`.
- Lisa `validation_data`, et jälgida mudeli üldistamisvõimet. Soovi korral kasuta ka `EarlyStopping`-ut.
- Tagasta `history`, et vajadusel hiljem graafikuid joonistada.

**Kontrolli:**
- `history.history` sisaldab võtmeid `loss` ja `val_loss`.
- Treening ei lõppe koheselt veaga ning mudel õpib (loss väheneb).

**Valikud:** 150 epochi ja `batch_size=16`; validatsiooniks kasutatakse testhulka (`validation_data`), et jälgida üldistumist.


In [ ]:
def train_model(model, X_train, X_test, y_train, y_test, epochs=150):
    """Train the model using training and testing datasets."""
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=16,
        verbose=0,
    )
    return history




## 11) Mudeli hindamine

Kasuta sisseehitatud `model.evaluate` meetodit, et saada täpsuse väärtus testandmetel.

**Sinu ülesanne:**
- Konverteeri sisendid NumPy massiivideks (sama andmetüüp (dtype) mis treenimisel).
- Käivita `model.evaluate` ja tagasta täpsus (teine väärtus tuple'ist).
- Ära prindi midagi funktsiooni seest, et teste lihtsustada.

**Kontrolli:**
- Tagastatud väärtus jääb vahemikku 0...1.


In [ ]:
def evaluate_model(model, X_test, y_test):
    """Evaluate the model and return accuracy."""
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)
    _, evaluation_accuracy = model.evaluate(X_test, y_test, verbose=0)
    return float(evaluation_accuracy)




## 12) Ennustustel põhinev hindamine

Võrdle mudeli ennustusi tegelike siltidega käsitsi, et kinnitada `model.evaluate` tulemust.

**Sinu ülesanne:**
- Genereeri ennustused `model.predict` abil.
- Kasuta `np.argmax`, et teisendada nii ennustused kui ka tegelikud one-hot vektorid klassi indeksiteks.
- Arvuta täpsus `np.mean(predicted_classes == expected_classes)` või `accuracy_score`.

**Kontrolli:**
- Saadud täpsus on väga lähedal `model.evaluate` väljundile (lubatud väike erinevus).


In [ ]:
def evaluate_using_predictions(model, X_test, y_test):
    """Evaluate accuracy by comparing predictions and expected labels."""
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)
    probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = np.argmax(y_test, axis=1)
    return float(accuracy_score(y_true, y_pred))




## 13) Peaprogramm

Selles etapis seod kõik eelnevad funktsioonid ühtseks toruks. Idee on lihtne: kasuta iga eelmise sammu tulemust järgmise sisendina, kuni jõuad mudeli hindamiseni.

**Sinu ülesanne:**
- Pane paika, millises järjekorras funktsioone kutsud (andmete lugemine → kodeerimine → jaotamine → mudeli loomine → treenimine → hindamine).
- Täienda allolevat `main()` funktsiooni nii, et kõik vajalikud vahepealsed muutujad tekivad päriselt.
- Käivita `main()` otseselt (`python iris_ann_template.ipynb` eksportimine pole vajalik) või läbi juhitava keskkonna, et näha, kas kõik sammud toimivad.

**Kontrolli:**
- Peaprogramm töötab ilma käsitsi sekkumiseta.
- Väljundis kuvatakse mõlemad täpsused ning mudeli konfiguratsiooni ülevaade (kui soovid).


In [ ]:
def main():
    """Run the full pipeline."""
    try_to_mount_drive()
    tf, _, _, _, _ = _get_tf_keras()

    np.random.seed(RANDOM_STATE)
    tf.random.set_seed(RANDOM_STATE)
    tf.keras.backend.clear_session()

    iris_file = resolve_iris_data_path()
    data = read_data(iris_file)

    encoded_data = encode_labels_to_numerical(data)
    X, y = separate_features_and_labels(encoded_data)
    y_one_hot = encode_labels_as_one_hot(y, num_classes=3)

    X_train, X_test, y_train, y_test = split_data_to_test_and_train(X, y_one_hot)
    X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

    model = create_model()
    model = compile_model(model)
    history = train_model(model, X_train_scaled, X_test_scaled, y_train, y_test, epochs=150)

    evaluation_accuracy = evaluate_model(model, X_test_scaled, y_test)
    prediction_accuracy = evaluate_using_predictions(model, X_test_scaled, y_test)

    return {
        "evaluation_accuracy": evaluation_accuracy,
        "prediction_accuracy": prediction_accuracy,
        "history": history,
        "model": model,
        "X_test_scaled": X_test_scaled,
        "y_test": y_test,
        "data": data,
        "scaler": scaler,
    }



if __name__ == "__main__":
    results = main()
    print(f"Evaluation accuracy: {results['evaluation_accuracy']:.4f}")
    print(f"Prediction accuracy: {results['prediction_accuracy']:.4f}")


## 14) Treeningu käik ja segadusmaatriks

Testhulk on väike (30 protsenti); käo kõver näitab, kas treening- ja valideerimiskad lähenevad — visuaalne kontroll üldistumise kohta.


In [ ]:
def show_training_report(results):
    """Show loss curves and confusion matrix manually."""
    import matplotlib.pyplot as plt

    hist = results["history"].history
    plt.figure(figsize=(8, 4))
    plt.plot(hist["loss"], label="train loss")
    plt.plot(hist["val_loss"], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.title("Mudeli kao (categorical_crossentropy)")
    plt.tight_layout()
    plt.show()

    y_prob = results["model"].predict(results["X_test_scaled"], verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.argmax(results["y_test"], axis=1)
    print("Confusion matrix (readout rows = true class, columns = predicted):")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))




## 15) Arutelu

**Kuidas erines närvivõrgu kasutamine varasematest klassifikatsioonimeetoditest?**
- Varasemates sklearn ülesannetes (nt loogiline regressioon, puud, KNN) määratakse sageli käsitsi või puustruktuuriga funktsioonide klassid; närvivõrk õpib mitmekihilise võrguga mittelineaarseid kombinatsioone otse andmetest gradiendi laskumise kaudu.
- Siin kasutatakse Keras/TensorFlow graafikutreeningut (`compile` + `fit`), mitte ainult `.fit`-lihtsat API-d ühes klõpsus ilma närvivõrgukihita.

**Millised on tehisnärvivõrkude eelised ja piirangud selle andmestiku puhul?**
- Eelised: paindlik mittelineaarne piirjoon; softmax annab tõenäosused klasside kohta; väike võrk treenib kiiresti.
- Piirangud: Iris on väike ja peaaegu lineaarselt eraldutav — sügavad võrgud riskeerivad ülearu sobimisega; tõlgendatavus on kehvem kui näiteks otsustuspuul.

**Kas lihtsama või keerukama mudeli kasutamine oleks siin põhjendatud?**
- Lihtsam mudel (vähesed Dense kihid, mõõdukas neuronite arv) on Irisel mõistlik: andmed on väikesed ja struktuur selge; keerukam võrk annaks vähe lisaväärtust ning võiks testimisel kõikuda rohkem.
